# 01. Playground для каскадного inference

Этот ноутбук нужен для быстрых экспериментов на одной картинке или небольшой папке картинок.

Что обычно меняют:

- thresholds в `config.yaml`
- `bad_class_names` в `config.yaml`
- NMS-флаги в `config.yaml`
- `IMAGE_PATH` ниже

In [ ]:
from pathlib import Path
import json
import sys

VALIDATOR_ROOT = Path.cwd()
if not (VALIDATOR_ROOT / "config.yaml").exists():
    VALIDATOR_ROOT = VALIDATOR_ROOT.parent

sys.path.insert(0, str(VALIDATOR_ROOT))
CONFIG_PATH = VALIDATOR_ROOT / "config.yaml"
print(VALIDATOR_ROOT)
print(CONFIG_PATH)


Загружаем каскад. Здесь создаются две ONNX Runtime session: сначала garment detector, затем bad-class detector.

In [ ]:
from src.cascade import CascadeFilter, load_settings

settings = load_settings(CONFIG_PATH)
print(settings)

cascade = CascadeFilter(settings)
print("cascade loaded")


Выбираем картинку. Если `IMAGE_PATH = None`, ноутбук возьмет первую картинку из split, указанного в конфиге.

In [ ]:
import yaml
from src.yolo_dataset import find_images

IMAGE_PATH = None

config = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
if IMAGE_PATH is None:
    dataset_root = Path(config["dataset"]["root"])
    split = config["dataset"].get("split", "val")
    IMAGE_PATH = find_images(dataset_root, split)[0]
else:
    IMAGE_PATH = Path(IMAGE_PATH)

print(IMAGE_PATH)


Запускаем каскад и смотрим сырой JSON-ответ.

In [ ]:
result = cascade.run_path(IMAGE_PATH)
print(json.dumps(result, ensure_ascii=False, indent=2))


Рисуем detections через matplotlib.

- синий bbox — garment prediction;
- желтый bbox — bad-class prediction, который прошел reject threshold;
- зеленый bbox — GT-разметка из YOLO label, если для картинки есть label-файл.

In [ ]:
from src.yolo_dataset import label_path_for, read_yolo_label_detections
from src.yolo_onnx import read_image_rgb
from src.visualization import draw_cascade_result_with_gt, show_image

image_rgb = read_image_rgb(IMAGE_PATH)
label_path = label_path_for(Path(config["dataset"]["root"]), config["dataset"].get("split", "val"), IMAGE_PATH)
gt_detections = read_yolo_label_detections(label_path)

drawn = draw_cascade_result_with_gt(image_rgb, result, gt_detections)
show_image(
    drawn,
    title=f'{result["decision"]}: {result["reason"]} / GT bbox: {len(gt_detections)}',
)


Опционально: прогоняем маленькую пачку картинок и показываем их сеткой. Это удобно для быстрой ручной проверки глазами.

In [ ]:
from src.visualization import show_images_grid

LIMIT = 8
dataset_root = Path(config["dataset"]["root"])
split = config["dataset"].get("split", "val")
images = find_images(dataset_root, split)[:LIMIT]
grid_items = []

for path in images:
    item = cascade.run_path(path)
    image_rgb = read_image_rgb(path)
    gt = read_yolo_label_detections(label_path_for(dataset_root, split, path))
    drawn = draw_cascade_result_with_gt(image_rgb, item, gt)
    title = f'{item["decision"]}: {item["reason"]}\nGT={len(gt)} pred={len(item["garment_detections"])} bad={len(item["bad_class_detections"])}\n{path.name}'
    grid_items.append((drawn, title))

show_images_grid(grid_items, columns=2)


Автоматически ищем и показываем примеры всех бинарных случаев: `TP`, `TN`, `FP`, `FN`.

Это самый полезный блок для ручной проверки правил каскада: сразу видно, где модель приняла правильное решение, а где ошиблась.

In [ ]:
from src.metrics import confusion_bucket
from src.yolo_dataset import iter_yolo_samples

TARGET_BUCKETS = ["TP", "TN", "FP", "FN"]
MAX_SEARCH = None  # можно поставить, например, 100 для быстрого smoke-test

dataset_root = Path(config["dataset"]["root"])
split = config["dataset"].get("split", "val")
samples = iter_yolo_samples(dataset_root, split, max_images=MAX_SEARCH)
examples = {}

for sample in samples:
    item = cascade.run_path(sample.image_path)
    bucket = confusion_bucket(item["decision"], sample.has_gt_bbox)
    if bucket not in examples:
        examples[bucket] = (sample, item)
    if all(bucket in examples for bucket in TARGET_BUCKETS):
        break

grid_items = []
for bucket in TARGET_BUCKETS:
    if bucket not in examples:
        print(f'Не найден пример {bucket}')
        continue

    sample, item = examples[bucket]
    image_rgb = read_image_rgb(sample.image_path)
    gt = read_yolo_label_detections(sample.label_path)
    drawn = draw_cascade_result_with_gt(image_rgb, item, gt)
    title = f'{bucket}: {item["decision"]} / {item["reason"]}\nGT={len(gt)} pred={len(item["garment_detections"])} bad={len(item["bad_class_detections"])}\n{sample.image_path.name}'
    grid_items.append((drawn, title))

show_images_grid(grid_items, columns=2)
